# Submission outcome diagnostics

This notebook keeps a session record for each Kaggle submission and compares successful, failed, pending, improved, and regressed runs. It combines Kaggle’s public submission result with the local Git revision, submission archive hash, local test evidence, and the change description.

A lower public score does not necessarily mean the agent never won: the score is matchmaking-based and depends on opponents, seat, randomness, and evaluation timing. The notebook identifies evidence and hypotheses rather than claiming a single cause.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess
from datetime import datetime, timezone
import pandas as pd

COMPETITION = 'pokemon-tcg-ai-battle'
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
OUT = ROOT / 'data' / 'submission_sessions'
OUT.mkdir(parents=True, exist_ok=True)

token = Path.home() / '.kaggle' / 'access_token'
if token.exists(): os.environ.setdefault('KAGGLE_API_TOKEN', token.read_text().strip())
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

def field(obj, *names, default=None):
    for name in names:
        value = getattr(obj, name, None)
        if value not in (None, ''): return value
    return default

def score(value):
    try: return float(value)
    except (TypeError, ValueError): return None

def run(*args):
    try: return subprocess.run(args, cwd=ROOT, text=True, capture_output=True, check=False).stdout.strip()
    except Exception: return ''

print('Repository:', ROOT)
print('Current commit:', run('git', 'rev-parse', '--short', 'HEAD'))

In [ ]:
rows = []
for item in api.competition_submissions(COMPETITION) or []:
    public = score(field(item, 'public_score', '_public_score'))
    status = str(field(item, 'status', '_status', default='')).split('.')[-1].upper()
    if public is not None: outcome = 'scored'
    elif 'ERROR' in status or field(item, 'error_description', '_error_description'): outcome = 'failed'
    else: outcome = 'pending'
    rows.append({
        'submission_id': field(item, 'ref', 'id', '_ref'),
        'submitted_at': str(field(item, 'date', '_date', default='')),
        'description': str(field(item, 'description', '_description', default='')),
        'status': status, 'outcome': outcome, 'public_score': public,
        'error': field(item, 'error_description', '_error_description'),
        'bytes': field(item, 'total_bytes', '_total_bytes'),
    })

df = pd.DataFrame(rows)
if len(df):
    df = df.sort_values('submitted_at', ascending=False).reset_index(drop=True)
    df['score_change_vs_previous'] = df['public_score'].diff(-1)
    df['result'] = df['public_score'].map(lambda x: 'not scored' if pd.isna(x) else ('above previous' if x > df.loc[0, 'public_score'] else 'scored'))
    display(df)
    print('Best scored run:', df.loc[df['public_score'].idxmax()].to_dict() if df['public_score'].notna().any() else 'none')
    print('Failure/pending count:', (df['outcome'] != 'scored').sum())
else:
    print('No submissions returned by Kaggle.')

In [ ]:
# Capture the exact local artifact and code state associated with this analysis.
tarball = ROOT / 'submission.tar.gz'
main = ROOT / 'submission' / 'main.py'
def sha256(path):
    if not path.exists(): return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

session = {
    'captured_at': datetime.now(timezone.utc).isoformat(),
    'competition': COMPETITION,
    'git_commit': run('git', 'rev-parse', 'HEAD'),
    'git_status': run('git', 'status', '--short'),
    'main_sha256': sha256(main),
    'submission_tar_sha256': sha256(tarball),
    'local_compile': subprocess.run(['python3', '-m', 'py_compile', str(main)], cwd=ROOT, capture_output=True, text=True).returncode == 0,
    'submissions': rows,
}
stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
(OUT / f'{stamp}.json').write_text(json.dumps(session, indent=2, ensure_ascii=False))
(OUT / 'latest.json').write_text(json.dumps(session, indent=2, ensure_ascii=False))
print('Session saved:', OUT / f'{stamp}.json')
print('Current main SHA-256:', session['main_sha256'])
print('Local compile passed:', session['local_compile'])

In [ ]:
# Explain likely causes for a regression using observable evidence.
scored = df[df['public_score'].notna()].copy() if len(df) else pd.DataFrame()
if len(scored) >= 2:
    scored['delta'] = scored['public_score'].diff(-1)
    for _, row in scored.iterrows():
        delta = row.get('delta')
        if pd.isna(delta): continue
        direction = 'improved' if delta > 0 else 'regressed' if delta < 0 else 'unchanged'
        print(f"{row['submission_id']}: {direction} by {delta:+.1f} — {row['description']}")
    print('\nInterpretation checklist:')
    print('- Verify the tarball hash and Git commit before attributing a score change to code.')
    print('- Compare local per-seat and matchup results; Kaggle’s matchmaking pool is different.')
    print('- Check for crashes, illegal actions, timeout pressure, and deck packaging differences.')
    print('- Treat one score as weak evidence; repeat converged submissions before changing policy.')
else:
    print('Need at least two scored submissions for regression analysis.')

## Current observation

A submission can win individual games and still receive a lower matchmaking score. The score is not a direct win count. Use the session files to tie each public result to the exact local commit, archive hash, description, and local validation result before drawing conclusions.